# PlantAdvisor: воспроизведение v3 и сохранение артефактов

Ноутбук повторяет обучение и оценку v3: тот же набор из 360/40 примеров, исходная модель и параметры QLoRA. Он предназначен для получения весов и полного комплекта результатов. Исходный выполненный прогон сохранён в `03_training_and_evaluation_v3.ipynb`; его выводы не перезаписываются.

Требуется Google Colab с T4. Данные загружаются автоматически из GitHub. Веса, контрольные точки и результаты этого запуска записываются в локальный каталог `/content/plant-advisor-v3-reproduction/`, отдельно от Drive и прежних экспериментов.

После обучения браузер скачает ZIP с адаптером и логами. После оценки — полный ZIP с ответами, оценками и графиками. Сохраните оба файла на компьютере: локальный диск Colab временный. Метрики нового прогона нужно проверять отдельно; их совпадение с предыдущими значениями не предполагается заранее.


## 1. Окружение

Зафиксированы версии библиотек и параметры GPU. Используются `transformers.Trainer` и PEFT; код вычисляет loss только по токенам ответа. В Colab требуется подключить Google Drive. Данные скачиваются автоматически, а результаты v3 сохраняются отдельно от предыдущих экспериментов.

Описание библиотек: [PEFT](https://huggingface.co/docs/peft/v0.18.0/en/developer_guides/quantization), [Trainer](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/trainer).


In [ ]:
%pip install -q --no-cache-dir "transformers==4.57.6" "peft==0.18.0" "accelerate==1.15.0" "bitsandbytes==0.50.2" "datasets==4.4.1" "huggingface-hub>=0.34,<1" "matplotlib>=3.7,<4" "pandas>=2,<3" "safetensors>=0.6,<1"

In [ ]:
from __future__ import annotations

import gc
import hashlib
import importlib.metadata
import json
import platform
import random
import re
import time
from collections import Counter
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from google.colab import drive
from IPython.display import display
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)
from transformers.trainer_utils import get_last_checkpoint


def load_pinned_tokenizer(repo_id: str, revision: str):
    """Загрузить токенизатор из закреплённой ревизии HF Hub.

    Если отсутствующий remote-каталог additional_chat_templates вызывает 404,
    загружаются только существующие файлы токенизатора.
    """
    try:
        return AutoTokenizer.from_pretrained(repo_id, revision=revision)
    except Exception as error:
        if "additional_chat_templates" not in str(error):
            raise

    from huggingface_hub import hf_hub_download
    from huggingface_hub.utils import EntryNotFoundError

    local_dir = Path("/content/plant_advisor_tokenizers") / (
        repo_id.replace("/", "_") + "_" + revision[:12]
    )
    local_dir.mkdir(parents=True, exist_ok=True)
    filenames = (
        "tokenizer.json", "tokenizer_config.json", "special_tokens_map.json",
        "config.json", "added_tokens.json", "vocab.json", "merges.txt",
        "chat_template.jinja",
    )
    for filename in filenames:
        try:
            hf_hub_download(
                repo_id=repo_id, revision=revision,
                filename=filename, local_dir=str(local_dir),
            )
        except EntryNotFoundError:
            continue
    print("Токенизатор загружен из локальных файлов:", local_dir)
    return AutoTokenizer.from_pretrained(local_dir, local_files_only=True)


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("Для обучения необходим Google Colab с GPU T4.")

torch.cuda.manual_seed_all(SEED)
drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/plant-advisor")

GITHUB_REPOSITORY = "Batyaoftheyear/plant-advisor"
GITHUB_REF = "main"  # Для точного повторения можно указать полный SHA коммита.


def load_github_sync():
    """Загрузить синхронизацию из того же публичного коммита, что и данные."""
    import importlib.util
    import os
    import subprocess
    from urllib.request import Request, urlopen

    if re.fullmatch(r"[0-9a-fA-F]{40}", GITHUB_REF):
        commit = GITHUB_REF.lower()
    else:
        result = subprocess.run(
            ["git", "-c", "credential.helper=", "-c", "core.askPass=",
             "-c", "http.extraHeader=", "ls-remote", "--exit-code",
             f"https://github.com/{GITHUB_REPOSITORY}.git",
             f"refs/heads/{GITHUB_REF}", f"refs/tags/{GITHUB_REF}",
             f"refs/tags/{GITHUB_REF}^{{}}"],
            check=True, capture_output=True, text=True, timeout=60,
            env={**os.environ, "GIT_TERMINAL_PROMPT": "0", "GIT_ASKPASS": ""},
        )
        refs = {line.split("\t", 1)[1]: line.split("\t", 1)[0]
                for line in result.stdout.splitlines() if "\t" in line}
        commit = (refs.get(f"refs/heads/{GITHUB_REF}")
                  or refs.get(f"refs/tags/{GITHUB_REF}^{{}}")
                  or refs.get(f"refs/tags/{GITHUB_REF}"))
    if not isinstance(commit, str) or not re.fullmatch(r"[0-9a-f]{40}", commit):
        raise ValueError("GitHub вернул некорректный SHA коммита.")
    helper_url = (f"https://raw.githubusercontent.com/{GITHUB_REPOSITORY}/"
                  f"{commit}/scripts/sync_colab_data.py")
    request = Request(helper_url, headers={"User-Agent": "PlantAdvisor-Colab-v3"})
    with urlopen(request, timeout=60) as response:
        helper_bytes = response.read(1024 * 1024)
    helper_dir = Path("/content/plant-advisor-github") / commit
    helper_dir.mkdir(parents=True, exist_ok=True)
    helper_path = helper_dir / "sync_colab_data.py"
    helper_path.write_bytes(helper_bytes)
    spec = importlib.util.spec_from_file_location("plant_advisor_sync_v3", helper_path)
    if spec is None or spec.loader is None:
        raise ImportError("Не удалось загрузить модуль синхронизации.")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module, commit


sync_module, GITHUB_COMMIT = load_github_sync()
SYNC_RECEIPT = sync_module.sync_from_github(
    ROOT, repository=GITHUB_REPOSITORY, revision=GITHUB_COMMIT
)
print("Данные v3 синхронизированы из GitHub:", GITHUB_COMMIT)
DATA_DIR = ROOT / "data" / "v3"
BASELINE_DIR = ROOT / "results" / "baseline" / "qwen25_3b_512"
JUDGE_ROOT = ROOT / "results" / "judge"
RUN_ROOT = Path("/content/plant-advisor-v3-reproduction")
SFT_DIR = RUN_ROOT / "results" / "sft" / "qwen25_3b_qlora_dataset_v3_recovery"
ADAPTER_DIR = SFT_DIR / "adapter"
CHECKPOINT_DIR = SFT_DIR / "checkpoints"
EVAL_DIR = RUN_ROOT / "results" / "evaluation" / "qlora_dataset_v3_recovery"
for directory in (SFT_DIR, CHECKPOINT_DIR, EVAL_DIR):
    directory.mkdir(parents=True, exist_ok=True)


def read_jsonl(path: Path) -> list[dict[str, Any]]:
    """Прочитать JSONL. Пример: read_jsonl(Path('data/train.jsonl')) -> [dict, ...]."""
    if not path.is_file():
        raise FileNotFoundError(f"Не найден файл: {path}")
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines()
            if line.strip()]


def append_jsonl(path: Path, item: dict[str, Any]) -> None:
    """Добавить запись в JSONL. Пример: append_jsonl(path, {'id': 'q01'})."""
    with path.open("a", encoding="utf-8") as file:
        file.write(json.dumps(item, ensure_ascii=False) + "\n")


def write_json(path: Path, obj: dict[str, Any]) -> None:
    """Записать JSON. Пример: write_json(path, {'epochs': 3})."""
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")


def sha256_file(path: Path) -> str:
    """Вычислить SHA-256 файла; пример результата — строка из 64 символов."""
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def package_version(name: str) -> str:
    """Вернуть установленную версию пакета, например '4.57.6'."""
    return importlib.metadata.version(name)


ENVIRONMENT = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    **{name: package_version(name) for name in (
        "transformers", "peft", "accelerate", "bitsandbytes", "datasets",
        "huggingface-hub",
    )},
}
write_json(SFT_DIR / "environment.json", ENVIRONMENT)
print("GPU:", ENVIRONMENT["gpu"])
print("Каталог проекта:", ROOT)
display(pd.DataFrame(ENVIRONMENT.items(), columns=["Компонент", "Значение"]))

## 2. Данные и контрольный протокол

Манифест подтверждает версию v3, размеры и SHA-256 файлов. Проверяются 360 обучающих, 40 валидационных и 30 контрольных записей, схема диалогов и отсутствие точных пересечений с тестом.

Системная инструкция, ревизия базовой модели и параметры генерации читаются из конфигурации baseline. Контрольные вопросы и исходные ответы сопоставляются по ID, категории и тексту. Это сохраняет условия сравнения до и после обучения.


In [ ]:
data_manifest = sync_module.verify_local(ROOT)
if data_manifest["dataset_version"] != "v3":
    raise ValueError("Для этого эксперимента необходим датасет v3.")
metadata_v3 = read_jsonl(DATA_DIR / "sft_metadata.jsonl")
if len(metadata_v3) != 400 or not all(row["id"].startswith("SFT3-") for row in metadata_v3):
    raise ValueError("Метаданные не соответствуют датасету v3.")

TRAIN_PATH = DATA_DIR / "train.jsonl"
VALID_PATH = DATA_DIR / "validation.jsonl"
QUESTIONS_PATH = ROOT / "data" / "eval_questions.jsonl"
BASELINE_CONFIG_PATH = BASELINE_DIR / "config.json"
BASELINE_ANSWERS_PATH = BASELINE_DIR / "baseline_answers.jsonl"

train_rows = read_jsonl(TRAIN_PATH)
validation_rows = read_jsonl(VALID_PATH)
eval_questions = read_jsonl(QUESTIONS_PATH)
baseline_rows = read_jsonl(BASELINE_ANSWERS_PATH)

if not BASELINE_CONFIG_PATH.exists():
    raise FileNotFoundError(
        "Не найден baseline/config.json. Проверьте выполнение автоматической синхронизации в предыдущей ячейке."
    )
baseline_config = json.loads(BASELINE_CONFIG_PATH.read_text(encoding="utf-8"))
BASE_MODEL_ID = baseline_config["model_id"]
BASE_MODEL_REVISION = baseline_config["revision"]
SYSTEM_PROMPT = baseline_config["system_prompt"]
MAX_NEW_TOKENS = baseline_config["generation"]["max_new_tokens"]

assert BASE_MODEL_ID == "Qwen/Qwen2.5-3B-Instruct"
assert len(train_rows) == 360, f"Ожидалось 360 train, получено {len(train_rows)}"
assert len(validation_rows) == 40, f"Ожидалось 40 validation, получено {len(validation_rows)}"
assert len(eval_questions) == len(baseline_rows) == 30


def validate_dialogue(row: dict[str, Any]) -> None:
    """Проверить формат одной SFT-пары user → assistant; исключение при ошибке."""
    messages = row.get("messages")
    if not isinstance(messages, list) or len(messages) != 2:
        raise ValueError("В примере ожидаются два сообщения: user и assistant.")
    if [message.get("role") for message in messages] != ["user", "assistant"]:
        raise ValueError("Неверный порядок ролей в обучающей записи.")
    if not all(isinstance(message.get("content"), str) and message["content"].strip()
               for message in messages):
        raise ValueError("Обнаружено пустое сообщение.")


for item in train_rows + validation_rows:
    validate_dialogue(item)

questions_by_id = {item["id"]: item for item in eval_questions}
baseline_by_id = {item["id"]: item for item in baseline_rows}
assert len(questions_by_id) == len(baseline_by_id) == 30
for question_id, item in questions_by_id.items():
    baseline = baseline_by_id[question_id]
    if (item["question"], item["task"]) != (baseline["question"], baseline["task"]):
        raise ValueError(f"Контрольный вопрос {question_id} изменился с момента baseline.")

train_questions = {item["messages"][0]["content"].strip().casefold()
                   for item in train_rows + validation_rows}
test_questions = {item["question"].strip().casefold() for item in eval_questions}
if train_questions & test_questions:
    raise ValueError("Обнаружено точное пересечение обучающих и тестовых вопросов.")

DATA_HASHES = {
    "manifest_v3": sha256_file(DATA_DIR / "manifest.json"),
    "metadata_v3": sha256_file(DATA_DIR / "sft_metadata.jsonl"),
    "train": sha256_file(TRAIN_PATH),
    "validation": sha256_file(VALID_PATH),
    "questions": sha256_file(QUESTIONS_PATH),
    "baseline_answers": sha256_file(BASELINE_ANSWERS_PATH),
}
print("Обучение:", len(train_rows), "Валидация:", len(validation_rows))
print("Контрольных вопросов:", len(eval_questions))
print("Базовая модель:", BASE_MODEL_ID)
print("Ревизия:", BASE_MODEL_REVISION)
print("Максимум новых токенов при тестировании:", MAX_NEW_TOKENS)

## 3. Токенизация и функция потерь

Последовательность состоит из системной инструкции, вопроса и ответа. Метки токенов инструкции и вопроса равны `-100`; loss вычисляется только по ответу ассистента, включая его завершение. Лимит — 1024 токена на диалог. Слишком длинные записи вызывают ошибку, а не скрыто обрезаются.


In [ ]:
MAX_SEQUENCE_LENGTH = 1024

tokenizer = load_pinned_tokenizer(BASE_MODEL_ID, BASE_MODEL_REVISION)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


def tokenize_dialogue(row: dict[str, Any]) -> dict[str, list[int]]:
    """Токенизировать диалог и замаскировать prompt для SFT.

    Вход: {'messages': [{'role': 'user', ...}, {'role': 'assistant', ...}]}.
    Выход: {'input_ids': [...], 'attention_mask': [...], 'labels': [-100, ..., id]}.
    """
    question = row["messages"][0]["content"]
    answer = row["messages"][1]["content"]
    prompt_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    full_messages = prompt_messages + [{"role": "assistant", "content": answer}]

    prefix_ids = tokenizer.apply_chat_template(
        prompt_messages, tokenize=True, add_generation_prompt=True
    )
    full_ids = tokenizer.apply_chat_template(
        full_messages, tokenize=True, add_generation_prompt=False
    )
    if full_ids[:len(prefix_ids)] != prefix_ids:
        raise ValueError("Токенизация prompt не совпадает с началом полного диалога.")
    if len(full_ids) > MAX_SEQUENCE_LENGTH:
        raise ValueError(
            f"Длина {len(full_ids)} превышает лимит {MAX_SEQUENCE_LENGTH}; "
            "необходимо увеличить MAX_SEQUENCE_LENGTH."
        )
    if len(full_ids) <= len(prefix_ids):
        raise ValueError("После prompt нет токенов ответа ассистента.")

    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": [-100] * len(prefix_ids) + full_ids[len(prefix_ids):],
    }


train_tokenized = [tokenize_dialogue(row) for row in train_rows]
validation_tokenized = [tokenize_dialogue(row) for row in validation_rows]
train_dataset = Dataset.from_list(train_tokenized)
validation_dataset = Dataset.from_list(validation_tokenized)

lengths = [len(item["input_ids"]) for item in train_tokenized + validation_tokenized]
supervised_tokens = [sum(value != -100 for value in item["labels"])
                     for item in train_tokenized + validation_tokenized]
print("Длина последовательностей (min / median / max):",
      min(lengths), int(np.median(lengths)), max(lengths))
print("Токены ответа с loss (min / median / max):",
      min(supervised_tokens), int(np.median(supervised_tokens)), max(supervised_tokens))
assert min(supervised_tokens) > 0

# Простая проверка корректности маски на первом примере.
first = train_tokenized[0]
assert len(first["input_ids"]) == len(first["labels"]) == len(first["attention_mask"])
assert any(label == -100 for label in first["labels"])
assert any(label != -100 for label in first["labels"])

## 4. Настройки QLoRA

Базовые веса загружены в NF4 с двойным квантованием и вычислениями FP16. Обучаются LoRA-матрицы в проекциях внимания и MLP: `r=16`, `alpha=32`, dropout `0.10`. Выполнены 2 эпохи с learning rate `3e-5`; эффективный batch size — 8 за счёт накопления градиента.

Валидация и сохранение проводятся каждые 15 шагов. Контрольная точка выбирается по минимальному `eval_loss` на SFT-validation, а не по контрольным вопросам. Конфигурация и хеши данных проверяются перед повторным использованием адаптера.


In [ ]:
EPOCHS = 2
MICRO_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 3.0e-5
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.10

TRAINING_CONFIG = {
    "dataset_version": "v3",
    "data_manifest_sha256": DATA_HASHES["manifest_v3"],
    "base_model": BASE_MODEL_ID,
    "base_revision": BASE_MODEL_REVISION,
    "system_prompt_sha256": hashlib.sha256(SYSTEM_PROMPT.encode("utf-8")).hexdigest(),
    "data_sha256": {"train": DATA_HASHES["train"], "validation": DATA_HASHES["validation"]},
    "seq_length": MAX_SEQUENCE_LENGTH,
    "epochs": EPOCHS,
    "micro_batch_size": MICRO_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "lora_target_modules": [
        "q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"
    ],
    "seed": SEED,
    "quantization": "4bit_nf4_double_quant_fp16",
}
CONFIG_PATH = SFT_DIR / "training_config.json"
if CONFIG_PATH.exists():
    saved_config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
    if saved_config != TRAINING_CONFIG:
        raise RuntimeError(
            "Существующие контрольные точки относятся к другим данным или гиперпараметрам. "
            "Сохраните старый эксперимент и используйте отдельную папку SFT_DIR."
        )
else:
    write_json(CONFIG_PATH, TRAINING_CONFIG)

write_json(SFT_DIR / "github_sync.json", SYNC_RECEIPT)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    revision=BASE_MODEL_REVISION,
    quantization_config=quantization_config,
    device_map={"": 0},
    dtype=torch.float16,
)
base_model.config.use_cache = False
print("Загружена модель:", BASE_MODEL_ID)
print("Доступная память GPU:",
      round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "ГиБ")

## 5. Обучение и адаптер

Обучаются LoRA-матрицы; базовые веса остаются замороженными. При наличии локального завершённого адаптера с той же конфигурацией он загружается повторно. Иначе обучение начинается заново либо продолжается с локальной контрольной точки.

Выходные файлы находятся в `results/sft/qwen25_3b_qlora_dataset_v3_recovery/` внутри локального каталога этого запуска. Старые папки на Drive не заменяются.


In [ ]:
ADAPTER_WEIGHTS = ADAPTER_DIR / "adapter_model.safetensors"
ADAPTER_CONFIG = ADAPTER_DIR / "adapter_config.json"

if ADAPTER_WEIGHTS.exists() and ADAPTER_CONFIG.exists():
    print("Готовый адаптер найден. Обучение повторно не запускается.")
    model = PeftModel.from_pretrained(base_model, ADAPTER_DIR, is_trainable=False)
    model.eval()
else:
    model = prepare_model_for_kbit_training(
        base_model, use_gradient_checkpointing=True
    )
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=TRAINING_CONFIG["lora_target_modules"],
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=None,
        padding=True,
        label_pad_token_id=-100,
        pad_to_multiple_of=8,
    )
    train_args = TrainingArguments(
        output_dir=str(CHECKPOINT_DIR),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=MICRO_BATCH_SIZE,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        weight_decay=0.0,
        fp16=True,
        bf16=False,
        optim="paged_adamw_8bit",
        eval_strategy="steps",
        save_strategy="steps",
        eval_steps=15,
        save_steps=15,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=2,
        logging_strategy="steps",
        logging_steps=10,
        report_to="none",
        remove_unused_columns=False,
        label_names=["labels"],
        dataloader_pin_memory=False,
        seed=SEED,
        data_seed=SEED,
    )
    trainer = Trainer(
        model=model,
        args=train_args,
        train_dataset=train_dataset,
        eval_dataset=validation_dataset,
        data_collator=collator,
    )
    resume_checkpoint = get_last_checkpoint(str(CHECKPOINT_DIR))
    if resume_checkpoint:
        print("Продолжение с контрольной точки:", resume_checkpoint)
    train_output = trainer.train(resume_from_checkpoint=resume_checkpoint)
    trainer.model.save_pretrained(ADAPTER_DIR, safe_serialization=True)
    tokenizer.save_pretrained(ADAPTER_DIR)
    trainer.save_state()

    write_json(SFT_DIR / "train_metrics.json", train_output.metrics)
    pd.DataFrame(trainer.state.log_history).to_csv(
        SFT_DIR / "training_log.csv", index=False
    )
    write_json(SFT_DIR / "best_checkpoint.json", {
        "best_checkpoint": trainer.state.best_model_checkpoint,
        "best_metric": trainer.state.best_metric,
        "global_step": trainer.state.global_step,
    })
    model = trainer.model
    del trainer
    gc.collect()
    torch.cuda.empty_cache()
    model.eval()
    print("Адаптер сохранён:", ADAPTER_DIR)

# Во время генерации кеш внимания разрешён: градиенты не вычисляются.
model.config.use_cache = True
assert ADAPTER_WEIGHTS.exists() and ADAPTER_CONFIG.exists(), "Адаптер не сохранён."

## 6. Динамика loss

График строится по журналу текущего запуска. На нём сопоставляются обучающий и валидационный loss; итоговое качество консультаций проверяется отдельно. После этой ячейки адаптер и журналы упаковываются в ZIP и скачиваются через браузер, до длительного оценивания судьёй.


In [ ]:
LOG_PATH = SFT_DIR / "training_log.csv"
if LOG_PATH.exists():
    history_df = pd.read_csv(LOG_PATH)
    display(history_df.tail(12))

    fig, ax = plt.subplots(figsize=(9, 4.4))
    train_curve = history_df.dropna(subset=["loss"]) if "loss" in history_df else pd.DataFrame()
    eval_curve = (history_df.dropna(subset=["eval_loss"])
                  if "eval_loss" in history_df else pd.DataFrame())
    if not train_curve.empty:
        ax.plot(train_curve["step"], train_curve["loss"],
                label="Train loss", marker=".", markersize=3)
    if not eval_curve.empty:
        ax.plot(eval_curve["step"], eval_curve["eval_loss"],
                label="Validation loss", marker="o")
    ax.set_xlabel("Шаг оптимизации")
    ax.set_ylabel("Cross-entropy loss")
    ax.set_title("Обучение QLoRA: train и validation")
    ax.grid(alpha=0.25)
    ax.legend()
    plt.tight_layout()
    fig.savefig(SFT_DIR / "loss_curves.png", dpi=170)
    plt.show()
else:
    print("Нет training_log.csv. Для построения кривой требуется сохранённый лог.")

In [ ]:
import importlib.util
from urllib.request import Request, urlopen
from google.colab import files

export_url = (f"https://raw.githubusercontent.com/{GITHUB_REPOSITORY}/"
              f"{GITHUB_COMMIT}/scripts/export_submission_artifacts.py")
request = Request(export_url, headers={"User-Agent": "PlantAdvisor-artifacts"})
with urlopen(request, timeout=60) as response:
    export_source = response.read(1024 * 1024)
export_path = RUN_ROOT / "export_submission_artifacts.py"
export_path.write_bytes(export_source)
spec = importlib.util.spec_from_file_location("plant_advisor_export", export_path)
export_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(export_module)

training_bundle = Path("/content/plant_advisor_v3_reproduction_training.zip")
training_info = export_module.make_bundle(RUN_ROOT, ROOT, training_bundle)
print(json.dumps(training_info, ensure_ascii=False, indent=2))
files.download(str(training_bundle))


## 7. Ответы после SFT

Получены ответы на те же 30 вопросов с исходной системной инструкцией и ревизией модели. Декодирование жадное: `do_sample=False`, `num_beams=1`, не более 512 новых токенов. JSONL сохраняется после каждого вопроса; конфигурация связывает ответы с адаптером и обучающими данными.


In [ ]:
REQUIRED_SECTIONS = {
    "care": ["условия", "рекомендации"],
    "diagnosis": ["возможные причины", "что проверить", "что делать"],
}
ALL_SECTION_NAMES = set().union(*(set(value) for value in REQUIRED_SECTIONS.values()))


def normalize_heading(line: str) -> str:
    """Нормализовать заголовок. Пример: '## 1. **Условия:**' -> 'условия'."""
    result = line.strip().casefold().replace("ё", "е")
    result = re.sub(r"^#{1,6}\s*", "", result)
    result = re.sub(r"^\d{1,2}[.)]\s*", "", result)
    result = re.sub(r"[*_`]", "", result)
    return result.removesuffix(":").strip()


def check_format(answer: str, task: str) -> dict[str, Any]:
    """Проверить наличие, порядок и содержимое разделов.

    Пример: check_format('Условия\n- Свет\nРекомендации\n- Полив', 'care')
    -> {'format_ok': True, ...}.
    """
    required = REQUIRED_SECTIONS[task]
    lines = answer.splitlines()
    headings = [(idx, normalize_heading(line)) for idx, line in enumerate(lines)
                if normalize_heading(line) in ALL_SECTION_NAMES]
    relevant = [(idx, name) for idx, name in headings if name in required]
    found = [name for _, name in relevant]
    missing = [name for name in required if name not in found]
    duplicates = [name for name in required if found.count(name) > 1]
    empty = []
    for idx, name in relevant:
        next_idx = min((pos for pos, _ in headings if pos > idx), default=len(lines))
        if not any(re.search(r"\w", line) for line in lines[idx + 1:next_idx]):
            empty.append(name)
    correct_order = found == required
    return {
        "format_ok": not missing and not duplicates and correct_order and not empty,
        "missing_sections": missing,
        "duplicate_sections": duplicates,
        "empty_sections": empty,
        "correct_order": correct_order,
    }


assert check_format("Условия\n- Свет\nРекомендации\n- Полив", "care")["format_ok"]
assert not check_format("Рекомендации\n- Полив\nУсловия\n- Свет", "care")["format_ok"]

SFT_ANSWERS_PATH = EVAL_DIR / "sft_answers.jsonl"
GENERATION_CONFIG = {
    "dataset_version": "v3",
    "training_config_sha256": sha256_file(CONFIG_PATH),
    "model": BASE_MODEL_ID,
    "base_revision": BASE_MODEL_REVISION,
    "adapter_data_hash": DATA_HASHES["train"],
    "system_prompt_sha256": TRAINING_CONFIG["system_prompt_sha256"],
    "generation": baseline_config["generation"],
    "questions_sha256": DATA_HASHES["questions"],
}
GENERATION_CONFIG_PATH = EVAL_DIR / "sft_generation_config.json"
if GENERATION_CONFIG_PATH.exists():
    if json.loads(GENERATION_CONFIG_PATH.read_text(encoding="utf-8")) != GENERATION_CONFIG:
        raise RuntimeError("Уже существуют SFT-ответы с другой конфигурацией.")
else:
    write_json(GENERATION_CONFIG_PATH, GENERATION_CONFIG)

sft_cache = {}
if SFT_ANSWERS_PATH.exists():
    for row in read_jsonl(SFT_ANSWERS_PATH):
        if row["id"] in sft_cache or row["id"] not in questions_by_id:
            raise ValueError(f"Некорректный ID в SFT-ответах: {row['id']}")
        sft_cache[row["id"]] = row


@torch.inference_mode()
def generate_sft_answer(question: str) -> dict[str, Any]:
    """Получить ответ по вопросу, вернув текст, токены и время генерации."""
    conversation = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    inputs = tokenizer.apply_chat_template(
        conversation, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device)
    start = time.perf_counter()
    generated = model.generate(
        **inputs, max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False, num_beams=1, pad_token_id=tokenizer.eos_token_id,
    )
    elapsed = time.perf_counter() - start
    new_tokens = generated[0, inputs["input_ids"].shape[-1]:]
    eos_ids = model.generation_config.eos_token_id
    eos_set = {eos_ids} if isinstance(eos_ids, int) else set(eos_ids or [])
    last_token = int(new_tokens[-1]) if len(new_tokens) else None
    return {
        "answer": tokenizer.decode(new_tokens, skip_special_tokens=True).strip(),
        "seconds": round(elapsed, 2),
        "generated_tokens": int(len(new_tokens)),
        "hit_token_limit": bool(len(new_tokens) >= MAX_NEW_TOKENS
                                and last_token not in eos_set),
    }


for index, item in enumerate(eval_questions, start=1):
    if item["id"] in sft_cache:
        continue
    result = generate_sft_answer(item["question"])
    record = {
        **item,
        **result,
        **check_format(result["answer"], item["task"]),
    }
    append_jsonl(SFT_ANSWERS_PATH, record)
    sft_cache[item["id"]] = record
    print(f"[{index:02d}/30] {item['id']} | формат: {record['format_ok']} | "
          f"токенов: {record['generated_tokens']}")

sft_rows = [sft_cache[item["id"]] for item in eval_questions]
assert len(sft_rows) == 30
print("Ответы сохранены:", SFT_ANSWERS_PATH)

## 8. Проверка формата

Одна функция рассчитывает FPR исходной и дообученной модели по всем 30 вопросам. Результат этого запуска сохраняется в таблицах; заранее заданное улучшение не предполагается. Содержательные оценки рассчитываются отдельно.


In [ ]:
format_rows = []
for version, records in (("baseline", baseline_rows), ("sft", sft_rows)):
    for record in records:
        check = check_format(record["answer"], record["task"])
        format_rows.append({
            "model": version,
            "id": record["id"],
            "task": record["task"],
            "format_ok": check["format_ok"],
            "hit_token_limit": record.get("hit_token_limit", False),
            "generated_tokens": record.get("generated_tokens", None),
        })

format_df = pd.DataFrame(format_rows)
format_table = (format_df.groupby(["model", "task"], as_index=False)
                .agg(n=("id", "count"), fpr=("format_ok", "mean"),
                     hit_token_limit_rate=("hit_token_limit", "mean")))
overall_format = (format_df.groupby("model", as_index=False)
                  .agg(n=("id", "count"), fpr=("format_ok", "mean")))
format_df.to_csv(EVAL_DIR / "format_by_question.csv", index=False)
format_table.to_csv(EVAL_DIR / "format_by_task.csv", index=False)
overall_format.to_csv(EVAL_DIR / "format_overall.csv", index=False)

display(overall_format.round(3))
display(format_table.round(3))

## 9. Единая модель-судья

Обе версии ответа оценены заново моделью `Qwen2.5-7B-Instruct`. Сохранены ревизия и основа промпта первого этапа; требования к полям диагностики уточнены. Поэтому оценки этого прогона не смешиваются с первыми 29 оценками baseline.

Критерии: корректность и применимость по шкале 1–5, диагностическое рассуждение для `diagnosis` и флаг серьёзной ошибки. На каждый случай допускаются до трёх попыток получения валидного JSON. Судья получает категорию, вопрос и ответ без обозначения оцениваемой модели.


In [ ]:
# До загрузки 7B освобождаем GPU от 3B-модели и LoRA.
del model, base_model, tokenizer
if "train_dataset" in globals():
    del train_dataset, validation_dataset, train_tokenized, validation_tokenized
gc.collect()
torch.cuda.empty_cache()
print("GPU после выгрузки SFT-модели:",
      round(torch.cuda.memory_allocated(0) / 1024**3, 2), "ГиБ")

JUDGE_CONFIG_CANDIDATES = [
    JUDGE_ROOT / "qwen25_7b_baseline_v2" / "config.json",
    JUDGE_ROOT / "qwen25_7b_baseline" / "config.json",
]
judge_config_path = next((path for path in JUDGE_CONFIG_CANDIDATES if path.exists()), None)
if judge_config_path is None:
    raise FileNotFoundError(
        "Не найдена конфигурация модели-судьи из первого ноутбука. "
        "Нужен results/judge/qwen25_7b_baseline_v2/config.json."
    )

saved_judge_config = json.loads(judge_config_path.read_text(encoding="utf-8"))
JUDGE_MODEL_ID = saved_judge_config["judge_model"]
JUDGE_REVISION = saved_judge_config["judge_revision"]
JUDGE_PROMPT = saved_judge_config["system_prompt"] + (
    "\nОБЯЗАТЕЛЬНО: при типе diagnosis укажите целое diagnostic_reasoning "
    "от 1 до 5 и непустой текст reasoning.diagnostic_reasoning. "
    "Для diagnosis значение null в этих полях ЗАПРЕЩЕНО. "
    "Только при care оба поля diagnostic_reasoning должны быть null. "
    "Верните только валидный JSON, без Markdown."
)
MAX_JUDGE_TOKENS = saved_judge_config["generation"]["max_new_tokens"]
JUDGE_MAX_ATTEMPTS = 3
assert JUDGE_MODEL_ID == "Qwen/Qwen2.5-7B-Instruct"

print("Конфигурация судьи:", judge_config_path)
print("Модель:", JUDGE_MODEL_ID)
print("Лимит токенов:", MAX_JUDGE_TOKENS)

judge_tokenizer = load_pinned_tokenizer(JUDGE_MODEL_ID, JUDGE_REVISION)
judge_model = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL_ID,
    revision=JUDGE_REVISION,
    quantization_config=quantization_config,
    device_map={"": 0},
    dtype=torch.float16,
)
judge_model.eval()

## 10. Оценки и пропуски

JSON судьи проверяется по типам, диапазонам и объяснениям. После исчерпания трёх попыток ошибка сохраняется в журнале. Такие вопросы не получают искусственных нулевых баллов; сводка показывает фактическое покрытие.


In [ ]:
def judge_input(item: dict[str, Any]) -> str:
    """Сформировать запрос для судьи с явно указанными типами полей."""
    if item["task"] == "diagnosis":
        instruction = (
            "diagnostic_reasoning: целое число 1..5, не null; "
            "reasoning.diagnostic_reasoning: непустая строка, не null."
        )
    else:
        instruction = (
            "diagnostic_reasoning: null; reasoning.diagnostic_reasoning: null."
        )
    return (
        f"Тип задачи: {item['task']}\n"
        f"Требования к полям: {instruction}\n"
        f"<QUESTION>\n{item['question']}\n</QUESTION>\n"
        f"<ANSWER>\n{item['answer']}\n</ANSWER>"
    )


def parse_judge_json(raw: str) -> dict[str, Any]:
    """Извлечь JSON из текста генерации; допускается Markdown-обрамление."""
    value = re.sub(r"^```(?:json)?\s*", "", raw.strip(), flags=re.I)
    value = re.sub(r"\s*```$", "", value).strip()
    try:
        item = json.loads(value)
        if isinstance(item, dict):
            return item
    except json.JSONDecodeError:
        pass
    decoder = json.JSONDecoder()
    for position, char in enumerate(value):
        if char != "{":
            continue
        try:
            item, _ = decoder.raw_decode(value[position:])
            if isinstance(item, dict):
                return item
        except json.JSONDecodeError:
            continue
    raise ValueError("В ответе судьи не найден корректный JSON-объект.")


def validate_judgment(item: dict[str, Any], task: str) -> dict[str, Any]:
    """Проверить типы, диапазоны оценок и наличие объяснений."""
    for key in ("factual_correctness", "actionability"):
        if type(item.get(key)) is not int or not 1 <= item[key] <= 5:
            raise ValueError(f"{key}: необходимо целое число от 1 до 5.")
    if task == "diagnosis":
        if type(item.get("diagnostic_reasoning")) is not int or not 1 <= item["diagnostic_reasoning"] <= 5:
            raise ValueError("Для diagnosis требуется diagnostic_reasoning от 1 до 5.")
    elif task == "care":
        if item.get("diagnostic_reasoning") is not None:
            raise ValueError("Для care diagnostic_reasoning должен быть null.")
    else:
        raise ValueError(f"Неизвестный тип задачи: {task}")
    if type(item.get("critical_error")) is not bool:
        raise ValueError("critical_error должен иметь тип bool.")
    reasons = item.get("reasoning")
    if not isinstance(reasons, dict):
        raise ValueError("Не найден блок reasoning.")
    for key in ("factual_correctness", "actionability"):
        if not isinstance(reasons.get(key), str) or not reasons[key].strip():
            raise ValueError(f"Нет объяснения для {key}.")
    if task == "diagnosis":
        if not isinstance(reasons.get("diagnostic_reasoning"), str) or not reasons["diagnostic_reasoning"].strip():
            raise ValueError("Нет пояснения для diagnostic_reasoning.")
    elif reasons.get("diagnostic_reasoning") is not None:
        raise ValueError("Для care пояснение diagnostic_reasoning должно быть null.")
    explanation = item.get("critical_error_explanation")
    if item["critical_error"]:
        if not isinstance(explanation, str) or not explanation.strip():
            raise ValueError("При critical_error=true требуется пояснение.")
    elif explanation is not None:
        raise ValueError("При critical_error=false пояснение должно быть null.")
    return item


example = {
    "factual_correctness": 4, "actionability": 5,
    "diagnostic_reasoning": None, "critical_error": False,
    "reasoning": {"factual_correctness": "Корректно.",
                  "actionability": "Есть действия.", "diagnostic_reasoning": None},
    "critical_error_explanation": None,
}
assert validate_judgment(parse_judge_json(json.dumps(example)), "care") == example


@torch.inference_mode()
def generate_judgment(item: dict[str, Any], correction: str = "") -> dict[str, Any]:
    """Выполнить оценку ответа, вернув сырой текст JSON и время генерации."""
    content = judge_input(item)
    if correction:
        content += ("\nПредыдущий ответ не прошёл проверку JSON: "
                    + correction[:200] + "\nВерни только корректный JSON.")
    messages = [
        {"role": "system", "content": JUDGE_PROMPT},
        {"role": "user", "content": content},
    ]
    inputs = judge_tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt",
    ).to(judge_model.device)
    started = time.perf_counter()
    generated = judge_model.generate(
        **inputs, max_new_tokens=MAX_JUDGE_TOKENS,
        do_sample=False, num_beams=1,
        pad_token_id=judge_tokenizer.eos_token_id,
    )
    elapsed = time.perf_counter() - started
    tokens = generated[0, inputs["input_ids"].shape[-1]:]
    return {"raw": judge_tokenizer.decode(tokens, skip_special_tokens=True).strip(),
            "seconds": round(elapsed, 2), "generated_tokens": int(len(tokens))}


JUDGE_RUN_CONFIG = {
    "judge_model": JUDGE_MODEL_ID,
    "judge_revision": JUDGE_REVISION,
    "prompt_sha256": hashlib.sha256(JUDGE_PROMPT.encode("utf-8")).hexdigest(),
    "input_template": "task + explicit diagnostic constraints + question + answer v2",
    "max_new_tokens": MAX_JUDGE_TOKENS,
    "do_sample": False,
    "baseline_sha256": sha256_file(BASELINE_ANSWERS_PATH),
    "sft_sha256": sha256_file(SFT_ANSWERS_PATH),
}
JUDGE_RUN_ID = hashlib.sha256(json.dumps(
    JUDGE_RUN_CONFIG, sort_keys=True, ensure_ascii=False
).encode("utf-8")).hexdigest()[:16]
JUDGE_DIR = EVAL_DIR / f"judge_{JUDGE_RUN_ID}"
JUDGE_DIR.mkdir(parents=True, exist_ok=True)
write_json(JUDGE_DIR / "config.json", JUDGE_RUN_CONFIG)


def judge_dataset(name: str, records: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """Оценить набор ответов, пропуская уже сохранённые успешные ID."""
    scores_path = JUDGE_DIR / f"{name}_scores.jsonl"
    raw_path = JUDGE_DIR / f"{name}_raw.jsonl"
    errors_path = JUDGE_DIR / f"{name}_failures.jsonl"
    completed = {item["id"]: item for item in read_jsonl(scores_path)} if scores_path.exists() else {}
    for index, record in enumerate(records, start=1):
        if record["id"] in completed:
            continue
        last_error = ""
        for attempt in range(1, JUDGE_MAX_ATTEMPTS + 1):
            result = generate_judgment(record, correction=last_error)
            append_jsonl(raw_path, {
                "id": record["id"], "attempt": attempt,
                "seconds": result["seconds"], "generated_tokens": result["generated_tokens"],
                "raw": result["raw"],
            })
            try:
                judgment = validate_judgment(
                    parse_judge_json(result["raw"]), record["task"]
                )
            except (ValueError, KeyError) as error:
                last_error = str(error)
                print(f"{name}, {record['id']}, попытка {attempt}: {last_error}")
                continue
            item = {
                "id": record["id"], "task": record["task"],
                "question": record["question"], "answer": record["answer"],
                **judgment,
            }
            append_jsonl(scores_path, item)
            completed[record["id"]] = item
            print(f"{name}: [{index:02d}/{len(records)}] {record['id']} | "
                  f"корректность={item['factual_correctness']}, "
                  f"полезность={item['actionability']}")
            break
        else:
            append_jsonl(errors_path, {"id": record["id"], "error": last_error})
            print(f"{name}: {record['id']} — не удалось получить оценку")
    return [completed[item["id"]] for item in records if item["id"] in completed]


# Оба набора оцениваются одним и тем же судьёй в одной сессии.
baseline_judgments = judge_dataset("baseline", baseline_rows)
sft_judgments = judge_dataset("sft", sft_rows)
print("Успешные оценки:", len(baseline_judgments), "baseline,", len(sft_judgments), "SFT")

## 11. Парное сравнение

Для содержательного сравнения используются только общие успешно оценённые ID. Перед сводкой выводятся выбранные пары ответов для ручного просмотра; этот список не является отдельной случайной выборкой. FPR рассчитывается по всем 30 вопросам.


In [ ]:
worse_ids = {
    "q01", "q03", "q04", "q05", "q08", "q09", "q12",
    "q14", "q17", "q18", "q22", "q26", "q30",
}

for baseline in baseline_rows:
  if baseline["id"] not in worse_ids:
        continue
  sft = next(
      (row for row in sft_rows if row["id"] == baseline["id"]),
      None,
  )

  if sft is None:
      continue

  print("=" * 100)
  print(f"ВОПРОС {baseline['id']}: {baseline['question']}")
  print(f"\nBASELINE:\n{baseline['answer']}")
  print(f"\nSFT:\n{sft['answer']}")
  print()

In [ ]:
baseline_judge_df = pd.DataFrame(baseline_judgments)
sft_judge_df = pd.DataFrame(sft_judgments)
if baseline_judge_df.empty or sft_judge_df.empty:
    raise RuntimeError("Нет достаточных оценок судьи для сравнения моделей.")

paired = baseline_judge_df.merge(
    sft_judge_df,
    on=["id", "task", "question"],
    suffixes=("_baseline", "_sft"),
    validate="one_to_one",
)
if paired.empty:
    raise RuntimeError("Судья не оценил ни одной общей пары ответов.")

score_fields = ["factual_correctness", "actionability", "diagnostic_reasoning"]
for metric in score_fields:
    paired[f"delta_{metric}"] = (
        paired[f"{metric}_sft"] - paired[f"{metric}_baseline"]
    )
paired.to_csv(EVAL_DIR / "paired_judge_scores.csv", index=False, encoding="utf-8-sig")


def summarize_pairs(group: pd.DataFrame, category: str) -> dict[str, Any]:
    """Агрегировать парные оценки и риск на поднаборе."""
    result = {"subset": category, "n_paired": len(group)}
    for metric in score_fields:
        result[f"{metric}_baseline"] = group[f"{metric}_baseline"].mean()
        result[f"{metric}_sft"] = group[f"{metric}_sft"].mean()
        result[f"{metric}_delta"] = group[f"delta_{metric}"].mean()
    for model_name in ("baseline", "sft"):
        result[f"critical_error_rate_{model_name}"] = (
            group[f"critical_error_{model_name}"].mean()
        )
    return result


comparison_df = pd.DataFrame([
    summarize_pairs(paired, "all"),
    summarize_pairs(paired[paired["task"] == "care"], "care"),
    summarize_pairs(paired[paired["task"] == "diagnosis"], "diagnosis"),
])
comparison_df.to_csv(EVAL_DIR / "comparison_by_task.csv", index=False)

fpr_index = overall_format.set_index("model")["fpr"]
summary = {
    "dataset_version": "v3",
    "github_commit": GITHUB_COMMIT,
    "n_questions": len(eval_questions),
    "n_judged_baseline": len(baseline_judgments),
    "n_judged_sft": len(sft_judgments),
    "n_paired": len(paired),
    "fpr_baseline": float(fpr_index["baseline"]),
    "fpr_sft": float(fpr_index["sft"]),
    "fpr_delta": float(fpr_index["sft"] - fpr_index["baseline"]),
    "judge_run_id": JUDGE_RUN_ID,
    "training_config_sha256": sha256_file(CONFIG_PATH),
    "data_hashes": DATA_HASHES,
}
write_json(EVAL_DIR / "comparison_summary.json", summary)
print("Общая сводка:")
display(pd.DataFrame([summary]).drop(columns=["data_hashes"]).T)
print("Сравнение по категориям (парная выборка):")
display(comparison_df.round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
models = ["baseline", "sft"]
axes[0].bar(models, [fpr_index["baseline"], fpr_index["sft"]])
axes[0].set_ylim(0, 1)
axes[0].set_title("Format Pass Rate (все 30 вопросов)")
axes[0].set_ylabel("Доля соблюдения формата")

main_row = comparison_df[comparison_df["subset"] == "all"].iloc[0]
positions = np.arange(2)
axes[1].bar(positions - 0.18, [main_row["factual_correctness_baseline"],
                                   main_row["actionability_baseline"]],
            width=0.35, label="Baseline")
axes[1].bar(positions + 0.18, [main_row["factual_correctness_sft"],
                                   main_row["actionability_sft"]],
            width=0.35, label="SFT")
axes[1].set_xticks(positions, ["Корректность", "Полезность"])
axes[1].set_ylim(0, 5)
axes[1].set_title(f"LLM-as-judge (парно: n={len(paired)})")
axes[1].set_ylabel("Средний балл (1–5)")
axes[1].legend()
plt.tight_layout()
fig.savefig(EVAL_DIR / "baseline_vs_sft.png", dpi=170)
plt.show()

print("\nСлучаи ухудшения фактической корректности:")
negative = paired[paired["delta_factual_correctness"] < 0]
display(negative[["id", "task", "delta_factual_correctness",
                  "factual_correctness_baseline", "factual_correctness_sft"]])

## 12. Выводы и выгрузка

Итоги определяются таблицами и примерами текущего запуска. Улучшение формата рассматривается отдельно от корректности и применимости. Следующая ячейка формирует полный архив с весами, конфигурациями, данными, логами, ответами и оценками.

Сохраните `plant_advisor_v3_reproduction_complete.zip` на компьютере. Для включения в репозиторий архив нужно поместить в рабочую папку проекта; он содержит контрольные суммы каждого файла. Этот прогон имеет отдельные пути и не подменяет результаты исходного выполненного ноутбука.


In [ ]:
complete_bundle = Path("/content/plant_advisor_v3_reproduction_complete.zip")
complete_info = export_module.make_bundle(
    RUN_ROOT, ROOT, complete_bundle, include_evaluation=True
)
print(json.dumps(complete_info, ensure_ascii=False, indent=2))
files.download(str(complete_bundle))
